# 03. Sequence Extraction

Maps each transporter candidate gene ID to its protein sequence from the ITAG4.0 proteome, and writes a FASTA file for BLAST search against Arabidopsis.

**Input**: `dataset/processed/final_candidates.csv`, `dataset/reference/ITAG4.0_proteins.fasta`

**Output**: `dataset/processed/final_candidates_with_seq.csv`, `dataset/blast/tomato_candidates.fasta`

In [30]:
from Bio import SeqIO
from Bio import Blast

import scanpy as sc
import pandas as pd
import numpy as np

In [11]:
candidates = pd.read_csv("../dataset/processed/final_candidates.csv")
fasta_dict = SeqIO.to_dict(SeqIO.parse("../dataset/reference/ITAG4.0_proteins.fasta", "fasta"))
fasta_dict

{'Solyc00g500001.1.1': SeqRecord(seq=Seq('HGTVILLVYVDDIIITGFTAALIEDVTRAMHTTFKMKDLGSLHYFLGMEVSRIG...TMW'), id='Solyc00g500001.1.1', name='Solyc00g500001.1.1', description='Solyc00g500001.1.1', dbxrefs=[]),
 'Solyc00g500002.1.1': SeqRecord(seq=Seq('MPISPSRSTNSVDVPFHNPRLFHSLVGGLQYLTVTRPDIQFAVNYVAQKMHSLT...SEV'), id='Solyc00g500002.1.1', name='Solyc00g500002.1.1', description='Solyc00g500002.1.1', dbxrefs=[]),
 'Solyc00g500003.1.1': SeqRecord(seq=Seq('MTSPPWTARTVRRRRAWHDITAPWTARTVELRRAWHDITAFGQHTRSTDVGRGM...PW*'), id='Solyc00g500003.1.1', name='Solyc00g500003.1.1', description='Solyc00g500003.1.1', dbxrefs=[]),
 'Solyc00g500004.1.1': SeqRecord(seq=Seq('MACHARRSRLCALSKSGDAMPRPTLPSVCAVQGGDVMPRPMLPIAMMEWNARRS...RR*'), id='Solyc00g500004.1.1', name='Solyc00g500004.1.1', description='Solyc00g500004.1.1', dbxrefs=[]),
 'Solyc00g500005.1.1': SeqRecord(seq=Seq('MPSPPLDSTHDRMTSGVGCHHRPSTMHTEGQRRAWHDLTALGQHTRSDYIGHGM...IE*'), id='Solyc00g500005.1.1', name='Solyc00g500005.1.1', description='Solyc00g5

In [12]:
candidates.head()

,names,logfoldchanges,pvals_adj,description,n_cell_types,cell_types,selected_as
0,Solyc05g006640.4,1.772674,4.101606e-15,Phospholipid-transporting ATPase (AHRD V3.3 **...,2,"['Epidermal', 'Mesophyll']",['mesophyll']
1,Solyc02g082450.3,2.163782,3.950407e-08,Auxin efflux carrier (AHRD V3.3 *** A0A200PYL2...,2,"['Epidermal', 'Mesophyll']",['mesophyll']
2,Solyc02g092450.3,3.237723,1.787920e-02,Calcium-transporting ATPase (AHRD V3.3 *** A0A...,1,['Mesophyll'],['mesophyll']
3,Solyc05g006020.3,7.455045,4.176399e-02,Nucleobase-ascorbate transporter 8 (AHRD V3.3 ...,1,['Mesophyll'],['mesophyll']
4,Solyc07g063520.3,0.872625,1.246746e-08,Organic solute transporter ostalpha protein (D...,3,"['Guard cell', 'Unknown', 'Vascular']",['gaurd']
5,Solyc11g062020.2,1.791141,9.387556e-05,Zinc transporter ZTP29 (AHRD V3.3 *** A0A2G3B9...,2,"['Epidermal', 'Vascular']",['vascular']
6,Solyc01g006150.3,0.754572,5.551576e-03,Cation efflux protein (AHRD V3.3 *** A0A200PY7...,2,"['Epidermal', 'Vascular']",['epidermal']
7,Solyc10g051120.3,1.621479,1.737716e-02,Mitochondrial pyruvate carrier (AHRD V3.3 *** ...,2,"['Epidermal', 'Vascular']",['vascular']
8,Solyc01g106900.3,1.289751,2.896952e-02,Magnesium transporter MRS2-F (AHRD V3.3 *** A0...,1,['Vascular'],['vascular']
9,Solyc09g082890.2,2.456515,1.838875e-05,Calcium-transporting ATPase (AHRD V3.3 *-* A0A...,2,"['Epidermal', 'Guard cell']",['gaurd']


In [ ]:
# Convert SeqRecord objects to plain strings for direct use in downstream string operations
id2seq = {k: str(v.seq) for k, v in fasta_dict.items()}

In [17]:
# ITAG4.0 proteome FASTA IDs carry an extra version suffix not present in the candidate gene IDs, so it is stripped before mapping
id2seq_trimmed = {k.rsplit(".", 1)[0]: v for k, v in id2seq.items()}
candidates["seq"] = candidates["names"].map(id2seq_trimmed)

In [18]:
candidates.head()

,names,logfoldchanges,pvals_adj,description,n_cell_types,cell_types,selected_as,seq
0,Solyc05g006640.4,1.772674,4.101606e-15,Phospholipid-transporting ATPase (AHRD V3.3 **...,2,"['Epidermal', 'Mesophyll']",['mesophyll'],MRNRIASSKNIRLGKVQPQAPGHRTVFCNDRDANSLAKFKGNSVST...
1,Solyc02g082450.3,2.163782,3.950407e-08,Auxin efflux carrier (AHRD V3.3 *** A0A200PYL2...,2,"['Epidermal', 'Mesophyll']",['mesophyll'],MGFMTLLEVASMPILQVLIVSGLGAVMATDYLKLLPADARKYLNRI...
2,Solyc02g092450.3,3.237723,1.787920e-02,Calcium-transporting ATPase (AHRD V3.3 *** A0A...,1,['Mesophyll'],['mesophyll'],MESYLNENFGGVKAKHSEEEMLRRWRSLCGVVKNPKRRFRFTANLS...
3,Solyc05g006020.3,7.455045,4.176399e-02,Nucleobase-ascorbate transporter 8 (AHRD V3.3 ...,1,['Mesophyll'],['mesophyll'],MAGGGAAAPPKQDENIPHPVKDQLPNVSYCITSPPPWPEAILLGFQ...
4,Solyc07g063520.3,0.872625,1.246746e-08,Organic solute transporter ostalpha protein (D...,3,"['Guard cell', 'Unknown', 'Vascular']",['gaurd'],MELDLDRGQVTLIASSICLMLTLHFSIQLVTEHFTSWKKPKEQKAI...
5,Solyc11g062020.2,1.791141,9.387556e-05,Zinc transporter ZTP29 (AHRD V3.3 *** A0A2G3B9...,2,"['Epidermal', 'Vascular']",['vascular'],MDSQVLVALGLSLLGGLSTSLGALFVVISQTPNLKMLGLLQGFAAG...
6,Solyc01g006150.3,0.754572,5.551576e-03,Cation efflux protein (AHRD V3.3 *** A0A200PY7...,2,"['Epidermal', 'Vascular']",['epidermal'],MEGEEIKNEVKTPLLDGWKLSYSGRRNSVTTLRGDFLARLPEKVKS...
7,Solyc10g051120.3,1.621479,1.737716e-02,Mitochondrial pyruvate carrier (AHRD V3.3 *** ...,2,"['Epidermal', 'Vascular']",['vascular'],MMKSVRAYLNSPMGPKTTHFWGPMANWGFVISGMMDTKKTPDAISG...
8,Solyc01g106900.3,1.289751,2.896952e-02,Magnesium transporter MRS2-F (AHRD V3.3 *** A0...,1,['Vascular'],['vascular'],MRGVTPPQKFSVPSPDDDADIIRPNTVIPGGVTVGAAMRKKATGVR...
9,Solyc09g082890.2,2.456515,1.838875e-05,Calcium-transporting ATPase (AHRD V3.3 *-* A0A...,2,"['Epidermal', 'Guard cell']",['gaurd'],MMFQANLHYLCMHITLDLPSEVDFSSKKRWHLAFATIYCSRAFEIK...


In [20]:
# Confirm every candidate gene ID was successfully mapped to a sequence before proceeding to BLAST
candidates["seq"].isna().sum()

np.int64(0)

In [21]:
candidates.to_csv("../dataset/processed/final_candidates_with_seq.csv", index=False)

In [33]:
# Trailing "*" marks the stop codon in ITAG protein sequences and must be removed for BLAST input
with open("../dataset/blast/tomato_candidates.fasta", "w") as f:
    for name, seq in zip(candidates["names"], candidates["seq"]):
        seq = seq.rstrip("*")
        f.write(f">{name}\n{seq}\n")